# 04 · Modelos preentrenados (encoders)

Encoder: **BETO** (`dccuchile/bert-base-spanish-wwm-cased`).
Se usan las mismas particiones `train` / `val` que el resto del TP. `test` no se toca hasta la evaluación final.
Métrica: Macro-F1 sobre `majority_label` con las clases `favor`, `against`, `neutral`.

## 1. Configuración, semillas y versiones

In [1]:
import os, random, time
import numpy as np, pandas as pd, torch, transformers, sklearn

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = "cuda" if torch.cuda.is_available() else "cpu"
def sync():  # para medir tiempos de GPU correctamente
    if device == "cuda": torch.cuda.synchronize()

print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)

C:\Users\juani\.venvs\tp3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda NVIDIA GeForce RTX 3050
torch 2.14.1+cu126 | transformers 5.22.2 | sklearn 1.9.1


## 2. Rutas (Colab con Drive o local)

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/TP IA y Neuro"   # ajustar si la carpeta está en otro lugar
except ImportError:
    BASE = "."                                      # ejecución local
DATA, RES = f"{BASE}/data", f"{BASE}/resultados"
os.makedirs(RES, exist_ok=True)
print(os.path.abspath(BASE))

C:\Users\juani\OneDrive\Escritorio\inteligencia-artificial-y-neurociencias\tp\3


## 3. Datos (solo train y val)

In [3]:
train = pd.read_csv(f"{DATA}/train.csv")
val   = pd.read_csv(f"{DATA}/val.csv")
# test.csv NO se carga hasta la evaluación final conjunta

LABELS = ["favor", "against", "neutral"]
# Los comentarios sin etiqueta mayoritaria quedan vacíos (NaN) al leer el CSV: se descartan
train = train[train["majority_label"].isin(LABELS)].reset_index(drop=True)
val   = val[val["majority_label"].isin(LABELS)].reset_index(drop=True)
print(train.shape, val.shape)
print(train["majority_label"].value_counts(), "\n", val["majority_label"].value_counts())
assert set(train["majority_label"]) <= set(LABELS) and set(val["majority_label"]) <= set(LABELS)
assert not (set(train["id"]) & set(val["id"])), "Hay ids repetidos entre train y val"

(1995, 9) (428, 9)
majority_label
favor      1246
against     439
neutral     310
Name: count, dtype: int64 
 majority_label
favor      267
against     94
neutral     67
Name: count, dtype: int64


## 4. Cargar BETO y medir longitudes

In [4]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

with torch.no_grad():
    enc = tokenizer(train["text"].iloc[0], return_tensors="pt", truncation=True).to(device)
    print("Salida:", encoder(**enc).last_hidden_state.shape)

lens = pd.Series([len(tokenizer(t)["input_ids"]) for t in train["text"]])
print(lens.describe(percentiles=[.5, .9, .95, .99]))
print("Proporción > 256 tokens:", round((lens > 256).mean(), 4))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19606.06it/s]


[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Salida: torch.Size([1, 40, 768])


count    1995.000000
mean       86.724311
std        41.603446
min        23.000000
50%        75.000000
90%       150.000000
95%       168.300000
99%       193.060000
max       254.000000
dtype: float64
Proporción > 256 tokens: 0.0


## 5. Sin fine-tuning: BETO congelado (mean pooling) + Regresión Logística

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report, confusion_matrix

MAX_LEN = 256

@torch.no_grad()
def embed(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        b = tokenizer(texts[i:i+bs], padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt").to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
            h = encoder(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).float()
        out.append(((h.float() * m).sum(1) / m.sum(1)).cpu().numpy())  # mean pooling
    return np.vstack(out)

set_seed()
# Entrenamiento = extraer embeddings de train + ajustar el clasificador
t0 = time.perf_counter()
X_tr = embed(train["text"].tolist())
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
clf.fit(X_tr, train["majority_label"])
sync(); t_train = time.perf_counter() - t0

# Inferencia = extraer embeddings de val + predecir
t0 = time.perf_counter()
X_va = embed(val["text"].tolist())
pred_frozen = clf.predict(X_va)
sync(); t_inf = time.perf_counter() - t0

f1 = f1_score(val["majority_label"], pred_frozen, labels=LABELS, average="macro")
print(f"Macro-F1 val: {f1:.4f} | entrenamiento {t_train:.1f}s | inferencia {t_inf:.1f}s")
print(classification_report(val["majority_label"], pred_frozen, labels=LABELS, digits=3))
print(confusion_matrix(val["majority_label"], pred_frozen, labels=LABELS))

# Mismas columnas que las tablas de resultados de los otros notebooks
resultados = [{
    "Modelo": "BETO congelado (mean pooling) + LogReg balanceada",
    "Macro-F1": f1,
    "Entrenamiento (s)": t_train,
    "Inferencia (s)": t_inf,
}]
pd.DataFrame(resultados)

Macro-F1 val: 0.4252 | entrenamiento 8.5s | inferencia 1.6s
              precision    recall  f1-score   support

       favor      0.718     0.610     0.660       267
     against      0.314     0.394     0.349        94
     neutral      0.241     0.299     0.267        67

    accuracy                          0.514       428
   macro avg      0.424     0.434     0.425       428
weighted avg      0.555     0.514     0.530       428

[[163  58  46]
 [ 40  37  17]
 [ 24  23  20]]


,Modelo,Macro-F1,Entrenamiento (s),Inferencia (s)
0,BETO congelado (mean pooling) + LogReg balanceada,0.425214,8.462448,1.571777


## 6. Con fine-tuning

Se entrena BETO completo con una capa de clasificación de 3 clases.
- Pérdida con pesos de clase *balanced* (igual criterio que los modelos clásicos).
- AdamW + scheduler lineal con 10% de warmup, precisión mixta (fp16), padding dinámico.
- **Tiempo de entrenamiento**: solo el loop de entrenamiento (no incluye la evaluación por época, que es solo de seguimiento).
- **Tiempo de inferencia**: tokenizar + predecir todo `val`.
- El Macro-F1 reportado es el del modelo al terminar la última época.

In [6]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

del encoder; torch.cuda.empty_cache()   # liberamos el encoder congelado

lab2id = {l: i for i, l in enumerate(LABELS)}
id2lab = {i: l for l, i in lab2id.items()}
y_tr = train["majority_label"].map(lab2id).values
y_va = val["majority_label"].map(lab2id).values

counts = np.bincount(y_tr, minlength=3)
class_w = torch.tensor(len(y_tr) / (3 * counts), dtype=torch.float, device=device)  # = class_weight="balanced"
print("Pesos de clase:", dict(zip(LABELS, class_w.cpu().numpy().round(3))))

def make_loader(texts, labels, bs, shuffle, max_len):
    data = list(zip(texts, labels))
    def collate(batch):
        t, y = zip(*batch)
        enc = tokenizer(list(t), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        enc["labels"] = torch.tensor(y)
        return enc
    return DataLoader(data, batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      generator=torch.Generator().manual_seed(SEED))

@torch.no_grad()
def predict(model, loader):
    model.eval(); preds = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items() if k != "labels"}
        with torch.autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
            preds.append(model(**b).logits.argmax(-1).cpu())
    return torch.cat(preds).numpy()

def fine_tune(lr=2e-5, epochs=3, max_len=256, bs=16, warmup=0.1, wd=0.01):
    set_seed()
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=id2lab, label2id=lab2id).to(device)
    tr_loader = make_loader(train["text"].tolist(), y_tr, bs, True, max_len)
    va_loader = make_loader(val["text"].tolist(), y_va, 64, False, max_len)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * len(tr_loader)
    sched = get_linear_schedule_with_warmup(opt, int(warmup * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_w)

    hist, t_train = [], 0.0
    for ep in range(1, epochs + 1):
        model.train(); sync(); t0 = time.perf_counter(); tot = 0.0
        for b in tr_loader:
            b = {k: v.to(device) for k, v in b.items()}
            y = b.pop("labels")
            with torch.autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
                logits = model(**b).logits
            loss = loss_fn(logits.float(), y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        sync(); t_train += time.perf_counter() - t0
        f_ep = f1_score(y_va, predict(model, va_loader), labels=[0, 1, 2], average="macro")
        hist.append({"época": ep, "loss train": tot / len(tr_loader), "Macro-F1 val": f_ep})
        print(f"época {ep}: loss {tot/len(tr_loader):.4f} | Macro-F1 val {f_ep:.4f}")

    sync(); t0 = time.perf_counter()
    pred = predict(model, va_loader)
    sync(); t_inf = time.perf_counter() - t0
    f1 = f1_score(y_va, pred, labels=[0, 1, 2], average="macro")
    print(f"FINAL -> Macro-F1 val {f1:.4f} | entrenamiento {t_train:.1f}s | inferencia {t_inf:.1f}s")
    print(classification_report(y_va, pred, labels=[0, 1, 2], target_names=LABELS, digits=3))
    print(confusion_matrix(y_va, pred, labels=[0, 1, 2]))
    return model, np.array([id2lab[i] for i in pred]), f1, t_train, t_inf, pd.DataFrame(hist)

Pesos de clase: {'favor': np.float32(0.534), 'against': np.float32(1.515), 'neutral': np.float32(2.145)}


### 6.1 Configuración base: lr = 2e-5, 3 épocas, max_len = 256

In [7]:
model_a, pred_a, f1_a, ttr_a, tinf_a, hist_a = fine_tune(lr=2e-5, epochs=3, max_len=256)
resultados.append({"Modelo": "BETO fine-tuning (lr=2e-5, 3 épocas, max_len=256)",
                   "Macro-F1": f1_a, "Entrenamiento (s)": ttr_a, "Inferencia (s)": tinf_a})
hist_a

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 15700.05it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:

época 1: loss 1.0543 | Macro-F1 val 0.5034


época 2: loss 0.8070 | Macro-F1 val 0.5300


época 3: loss 0.5311 | Macro-F1 val 0.5729


FINAL -> Macro-F1 val 0.5729 | entrenamiento 94.1s | inferencia 1.6s
              precision    recall  f1-score   support

       favor      0.788     0.655     0.716       267
     against      0.509     0.596     0.549        94
     neutral      0.385     0.552     0.454        67

    accuracy                          0.626       428
   macro avg      0.561     0.601     0.573       428
weighted avg      0.664     0.626     0.638       428

[[175  43  49]
 [ 28  56  10]
 [ 19  11  37]]


,época,loss train,Macro-F1 val
0,1,1.054259,0.503405
1,2,0.806952,0.530039
2,3,0.531058,0.572918


### 6.2 Variante: learning rate más alto (lr = 5e-5), resto igual

In [8]:
model_a = model_a.cpu(); torch.cuda.empty_cache()   # liberamos GPU
model_b, pred_b, f1_b, ttr_b, tinf_b, hist_b = fine_tune(lr=5e-5, epochs=3, max_len=256)
resultados.append({"Modelo": "BETO fine-tuning (lr=5e-5, 3 épocas, max_len=256)",
                   "Macro-F1": f1_b, "Entrenamiento (s)": ttr_b, "Inferencia (s)": tinf_b})
hist_b

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19468.86it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:

época 1: loss 1.0548 | Macro-F1 val 0.5544


época 2: loss 0.6687 | Macro-F1 val 0.5515


época 3: loss 0.2736 | Macro-F1 val 0.5431


FINAL -> Macro-F1 val 0.5431 | entrenamiento 94.4s | inferencia 1.6s
              precision    recall  f1-score   support

       favor      0.741     0.760     0.750       267
     against      0.564     0.564     0.564        94
     neutral      0.333     0.299     0.315        67

    accuracy                          0.645       428
   macro avg      0.546     0.541     0.543       428
weighted avg      0.638     0.645     0.641       428

[[203  30  34]
 [ 35  53   6]
 [ 36  11  20]]


,época,loss train,Macro-F1 val
0,1,1.054800,0.554393
1,2,0.668692,0.551471
2,3,0.273575,0.543084


## 7. Resultados y guardado

In [9]:
tabla = pd.DataFrame(resultados)
tabla.to_csv(f"{RES}/modelos_preentrenados.csv", index=False)

pd.DataFrame({"text": val["text"], "majority_label": val["majority_label"],
              "beto_congelado": pred_frozen, "beto_ft_lr2e-5": pred_a, "beto_ft_lr5e-5": pred_b}
             ).to_csv(f"{RES}/predicciones_preentrenados.csv", index=False)

pd.concat([hist_a.assign(config="lr=2e-5"), hist_b.assign(config="lr=5e-5")]
          ).to_csv(f"{RES}/curvas_finetuning.csv", index=False)

tabla.round(4)

,Modelo,Macro-F1,Entrenamiento (s),Inferencia (s)
0,BETO congelado (mean pooling) + LogReg balanceada,0.4252,8.4624,1.5718
1,"BETO fine-tuning (lr=2e-5, 3 épocas, max_len=256)",0.5729,94.1076,1.6430
2,"BETO fine-tuning (lr=5e-5, 3 épocas, max_len=256)",0.5431,94.4246,1.6480


## 8. Evaluación final en test (NO ejecutar hasta la evaluación conjunta)

Se elige la configuración con mejor Macro-F1 en **validación** y recién ahí se mide en `test`.
Para correrla, cambiar `EVALUAR_TEST = True`.

In [10]:
EVALUAR_TEST = False

if EVALUAR_TEST:
    best_model, best_name = (model_a, "lr=2e-5") if f1_a >= f1_b else (model_b, "lr=5e-5")
    best_model = best_model.to(device)
    test = pd.read_csv(f"{DATA}/test.csv")
    test = test[test["majority_label"].isin(LABELS)].reset_index(drop=True)
    y_te = test["majority_label"].map(lab2id).values
    te_loader = make_loader(test["text"].tolist(), y_te, 64, False, 256)
    sync(); t0 = time.perf_counter(); p_te = predict(best_model, te_loader); sync()
    print(f"TEST ({best_name}) Macro-F1: {f1_score(y_te, p_te, labels=[0,1,2], average='macro'):.4f} "
          f"| inferencia {time.perf_counter()-t0:.1f}s")
else:
    print("Test reservado: no se evaluó.")

Test reservado: no se evaluó.
